# Responses API vs Chat Completions API

In this course, we use the **Responses API** for OpenAI calls.

Many tutorials, blog posts, and older examples still use the **Chat Completions API**. This notebook gives you the minimum context you need so that both styles make sense.

## Learning goals
- Recognize the two OpenAI API styles in code.
- Explain why this course uses the Responses API.
- Translate a simple Chat Completions example into a Responses API example.
- Know when provider compatibility might matter.

## Prerequisites
- You have already made a first LLM call with the OpenAI Python SDK.
- You have an `OPENAI_API_KEY` in your `.env` file if you want to run the API cells.


## The short version

| Question | Short answer |
| --- | --- |
| Which API do we use in this course? | Responses API |
| Why? | It is OpenAI's newer, more capable interface for text, reasoning, tools, multimodal input, and state. |
| Is Chat Completions still useful? | Yes. You will still see it in older examples and in provider-compatible APIs. |
| Should you learn every difference now? | No. For the MVP, learn the basic shape and move on. |


## Setup

The code below loads the API key from `.env` and selects a small model by default. You can override the model by setting `OPENAI_MODEL` in your environment.

In [ ]:
from __future__ import annotations

import os
import textwrap

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI() if api_key else None

print(f"Model: {MODEL}")
if client is None:
    print(
        "No OPENAI_API_KEY found. You can still read the notebook, but API cells will be skipped."
    )
else:
    print("OPENAI_API_KEY found. API cells can run.")

We will use a tiny version of the job-digest task, because that is the project context of this course.

In [ ]:
job_posting = """
AI Engineer

We are looking for a Python developer who can build LLM-powered workflows.
The role includes prompt design, structured outputs, evaluations, and production monitoring.
""".strip()

prompt = f"""
Summarize this job posting in one concise sentence:

{job_posting}
""".strip()

print(prompt)

## Chat Completions API

Chat Completions is built around a list of `messages`. Each message has a `role` and `content`.

The response text is nested under `choices[0].message.content`.

In [ ]:
if client is None:
    print("Skipped because OPENAI_API_KEY is missing.")
else:
    chat_completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt},
        ],
    )

    chat_text = chat_completion.choices[0].message.content
    print(textwrap.fill(chat_text, width=88))

## Responses API

Responses can accept a simple string as `input`, which is enough for many first examples.

The response text is available as `response.output_text`.

In [ ]:
if client is None:
    print("Skipped because OPENAI_API_KEY is missing.")
else:
    response = client.responses.create(
        model=MODEL,
        input=prompt,
    )

    responses_text = response.output_text
    print(textwrap.fill(responses_text, width=88))

## What changed?

| Chat Completions | Responses |
| --- | --- |
| `client.chat.completions.create(...)` | `client.responses.create(...)` |
| Input is usually `messages=[...]` | Input can be a string or a list of input items |
| Text is under `choices[0].message.content` | Text is available as `output_text` |
| Common in older tutorials | Better default for new OpenAI projects |
| Useful when matching provider-compatible APIs | Better fit for OpenAI tools, reasoning, multimodal input, and state |


## Why this course uses Responses

We use Responses because it keeps the course moving forward:

- It is simple for basic text generation.
- It works well with reasoning models.
- It scales naturally to structured output and tool calling later in this module.
- It supports conversation chaining with `previous_response_id` when we need state.

The important point is not that Chat Completions is bad. The important point is that Responses is the better default for this course.

## Optional: conversation state

LLMs do not remember previous calls by default. With Chat Completions, you usually pass the full message history yourself.

With Responses, OpenAI also supports chaining calls with `previous_response_id`. This is useful later when you build conversational or tool-using workflows.

Keep this optional for now. The MVP job digest mostly uses independent workflow steps, not a long chat history.

In [ ]:
RUN_OPTIONAL_STATE_DEMO = False

if not RUN_OPTIONAL_STATE_DEMO:
    print("Optional demo skipped. Set RUN_OPTIONAL_STATE_DEMO = True to run it.")
elif client is None:
    print("Skipped because OPENAI_API_KEY is missing.")
else:
    first_response = client.responses.create(
        model=MODEL,
        input="Name one skill from the AI Engineer job posting.",
    )

    second_response = client.responses.create(
        model=MODEL,
        previous_response_id=first_response.id,
        input="Why is that skill useful for the job digest project?",
    )

    print("First response:")
    print(textwrap.fill(first_response.output_text, width=88))
    print("\nSecond response:")
    print(textwrap.fill(second_response.output_text, width=88))

## Exercise

Convert this Chat Completions call into a Responses call:

```python
chat_completion = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Extract three skills from this job posting."},
    ],
)

print(chat_completion.choices[0].message.content)
```

Try to fill in the next cell before looking at the answer.

In [ ]:
# Your turn: complete this Responses API call.
# Keep the same model and user task.

# response = client.responses.create(
#     model=...,
#     input=...,
# )
# print(...)

## Answer

The equivalent Responses call is:

In [ ]:
if client is None:
    print("Skipped because OPENAI_API_KEY is missing.")
else:
    response = client.responses.create(
        model=MODEL,
        input="Extract three skills from this job posting.",
    )

    print(response.output_text)

## Takeaway

Chat Completions is the older conversation-shaped endpoint you will still see in many examples. Responses is OpenAI's newer default for new projects, and that is why we use it in this course.

## Sources
- OpenAI migration guide: https://developers.openai.com/api/docs/guides/migrate-to-responses
- OpenAI text generation guide: https://developers.openai.com/api/docs/guides/text
- OpenAI conversation state guide: https://developers.openai.com/api/docs/guides/conversation-state
